In [ ]:
import time
from IPython.display import Markdown, display

# =========================
#  Utility Prompt Builders
# =========================

def build_basic_prompt(test_case_text):
    return f"""
You are an expert test automation engineer specializing in BDD and Cucumber.
Transform the following test cases into a complete, production-ready Cucumber test suite.

## INPUT TEST CASES:
{test_case_text}

## EXPECTED OUTPUT:

### FEATURE FILE (.feature)
- Clear feature name & description
- Background (if applicable)
- Scenarios for each test case
- Scenario Outlines & Examples (when needed)
- Tags like @smoke, @regression, @critical
- Proper Gherkin syntax (Given/When/Then/And/But)

### STEP DEFINITIONS (Java)
- Imports (Selenium WebDriver, Cucumber)
- @Given, @When, @Then annotations with regex
- Page Object references
- Explicit waits
- Clean error handling & logging
- Hooks (@Before, @After)
- Parameter handling and datatables

### BEST PRACTICES
- Atomic reusable steps
- Declarative language (business readable)
- DRY principle
- Clear variable names
- Assertions

### OUTPUT FORMAT:
**FEATURE FILE:**
```gherkin
[Feature file]
STEP DEFINITIONS (LoginSteps.java):
[Step definitions]
TEST DATA NOTES:
[Notes]
EXECUTION NOTES:
[Run instructions]
Generate the test suite now.
"""
def build_advanced_prompt(test_case_text, config):
framework_map = {
'selenium': 'Selenium WebDriver',
'appium': 'Appium',
'restassured': 'RestAssured',
'playwright': 'Playwright'
}
pattern_map = {
    'page_object': 'Page Object Model (POM)',
    'screenplay': 'Screenplay Pattern',
    'traditional': 'Traditional procedural Cucumber steps'
}

return f"""
You are an expert Cucumber + BDD automation engineer.
Generate a complete test suite based on the configuration below.
INPUT TEST CASES:
{test_case_text}
CONFIGURATION:
•	Framework: {framework_map.get(config['framework'], config['framework'])}
•	Language: {config['language'].upper()}
•	Pattern: {pattern_map.get(config['pattern'], config['pattern'])}
•	Tags: {', '.join(config['tags'])}
•	Include Hooks: {config['include_hooks']}
•	Include Example Tables: {config['include_examples']}
•	Include Negative Scenarios: {config['include_negative']}
EXPECTED OUTPUT:
FEATURE FILE
•	Feature description
•	Scenario Outlines (if enabled)
•	Tags
•	Background (if needed)
•	Negative scenarios (if enabled)
STEP DEFINITIONS
•	Proper imports
•	Parameterized regex steps
•	Error handling
•	Wait strategies
{ "- Include @Before/@After hooks" if config["include_hooks"] else "" }
SUPPORTING CODE
{ "- Page Objects with locators & actions" if config["pattern"] == "page_object" else "" }
{ "- Actor/Task/Question classes" if config["pattern"] == "screenplay" else "" }
BEST PRACTICES
•	Declarative steps
•	Explicit waits
•	Separate test data
•	Independent scenarios
OUTPUT FORMAT:
FEATURE FILE:
[Feature]
STEP DEFINITIONS:
[Steps]
{f"""PAGE OBJECTS:
[POM class]

**EXECUTION NOTES:**
[Run instructions]

Generate the suite now.
"""


# ===================================
#  BASIC CUCUMBER GENERATOR FUNCTION
# ===================================
def generate_cucumber_script(qa_chain, test_case_text, show_confidence=True):

    if not qa_chain:
        print("❌ QA chain not initialized.")
        return

    if not test_case_text.strip():
        print("❌ Test case text is empty.")
        return

    print("🥒 Generating Basic Cucumber Script...")
    prompt = build_basic_prompt(test_case_text)

    start = time.time()
    response = qa_chain.invoke({"query": prompt})
    end = time.time()

    print("\n🥒 Output:")
    display(Markdown(response['result']))

    if show_confidence:
        confidence_score = calculate_confidence_level(prompt, response["result"])
        match_score = calculate_match_percentage(response["result"], test_case_text)
        display_confidence_metrics(confidence_score, match_score)

    print(f"⏱️ Completed in {end - start:.2f}s")
    return response["result"]


# ============================================
#  ADVANCED CUCUMBER GENERATOR FUNCTION
# ============================================

def generate_cucumber_script_advanced(qa_chain, test_case_text, config=None):

    if not qa_chain:
        print("❌ QA chain not initialized.")
        return

    if not test_case_text.strip():
        print("❌ Test case text is empty.")
        return

    defaults = {
        'framework': 'selenium',
        'language': 'java',
        'pattern': 'page_object',
        'tags': ['@automated'],
        'include_hooks': True,
        'include_examples': True,
        'include_negative': True,
        'show_confidence': True
    }

    config = {**defaults, **(config or {})}

    print(f"🥒 Generating Advanced Script ({config['framework']} + {config['pattern']})...")
    prompt = build_advanced_prompt(test_case_text, config)

    start = time.time()
    response = qa_chain.invoke({"query": prompt})
    end = time.time()

    print("\n🥒 Output:")
    display(Markdown(response['result']))

    if config['show_confidence']:
        confidence_score = calculate_confidence_level(prompt, response["result"])
        match_score = calculate_match_percentage(response["result"], test_case_text)
        display_confidence_metrics(confidence_score, match_score)

    print(f"⏱️ Completed in {end - start:.2f}s")
    return response["result"]